# V5 Supplementary Investigation Displays

Supplementary exploratory displays and their original execution outputs are retained as a historical record. These cells are not part of the sequential V5 batch runner and may depend on state created in the original session.

In [3]:
# Investigation report inspection
from pathlib import Path
import json

REPORTS = Path(
    '/content/drive/MyDrive/GC1/Temporal_snapshots/'
    'agentic_graphrag_investigation_v5_full_139_output/'
    'llm_pilot_reports.jsonl'
)

assert REPORTS.exists(), f"Missing: {REPORTS}"

reports = [
    json.loads(line)
    for line in REPORTS.read_text(encoding='utf-8').splitlines()
    if line.strip()
]

print("Total investigation reports:", len(reports))

dependency_reports = [
    r for r in reports
    if 'dependency' in [
        str(x).lower() for x in r.get('risk_types', [])
    ]
    and r.get('conclusion') == 'LIMITED_EVIDENCE_OBSERVATIONS'
    and len(r.get('claims') or []) > 0
]

print("Dependency investigations with observations:",
      len(dependency_reports))

Total investigation reports: 139
Dependency investigations with observations: 43


In [5]:
from IPython.display import display, Markdown

assert dependency_reports, "No suitable dependency reports found"

report = dependency_reports[0]

print("Investigation ID:", report['trigger_id'])
print("Project instance:", report['instance_id'])
print("Checkpoint:", report['checkpoint_time'])
print("Risk types:", report['risk_types'])
print("Generation mode:", report['generation_mode'])

display(Markdown("### Investigation Summary"))
print(report.get('summary', ''))

display(Markdown("### Limitations"))

caveats = report.get('caveats', '')

if isinstance(caveats, str):
    print(caveats)
elif isinstance(caveats, list):
    for caveat in caveats:
        print("-", caveat)
else:
    print("No limitations provided.")

Investigation ID: 041a60f311c0263b395c
Project instance: ENTESB-I01
Checkpoint: 2020-08-26T07:32:26+00:00
Risk types: ['dependency', 'scope']
Generation mode: LOCAL_LLM


### Investigation Summary

There is limited evidence of active dependencies and status changes in JIRA issues related to project risks. Dependency chains and status transitions are observed but not conclusively linked to specific project risks.


### Limitations

The provided evidence is derived from JIRA status transitions and dependency chains. It does not establish causality or predict future outcomes.


In [10]:
from IPython.display import display, Markdown

# Select the previously identified investigation report
trigger_id = report['trigger_id']

# Collect evidence IDs cited by the LLM
cited_ids = set()

for claim in report.get('claims', []):
    cited_ids.update(claim.get('evidence_ids', []))

print("Investigation:", trigger_id)
print("Cited evidence IDs:", len(cited_ids))

# Locate the original investigation request
request = next(
    (r for r in chosen if r['trigger_id'] == trigger_id),
    None
)

assert request is not None, "Investigation request not found"

# Display the original evidence supplied to the LLM
evidence_records = request.get('evidence', [])

print("\nEvidence records in investigation:",
      len(evidence_records))

for e in evidence_records:
    evidence_id = e.get('evidence_id', '')

    if evidence_id in cited_ids:
        display(Markdown(f"### Evidence: `{evidence_id}`"))

        print("Source:", e.get('source_type', 'Unknown'))
        print("Source ID:", e.get('source_id', 'Unknown'))
        print("Timestamp:", e.get('available_at', 'Unknown'))
        print("Evidence text:", e.get('text', ''))

Investigation: 041a60f311c0263b395c
Cited evidence IDs: 4

Evidence records in investigation: 6


### Evidence: `DEP::ENTESB-I01::20::ENTESB-13841::ENTESB-13085`

Source: JIRA
Source ID: T_bff48bc9462b4ae925bf0953
Timestamp: 2020-05-28T07:51:08+00:00
Evidence text: Active Jira dependency: upstream issue ENTESB-13841 blocks downstream issue ENTESB-13085; link active since 2020-05-28T07:51:08+00:00.


### Evidence: `DEP::ENTESB-I01::20::ENTESB-13841::ENTESB-11455`

Source: JIRA
Source ID: T_57721b5d3e5ea74548301c20
Timestamp: 2020-05-28T07:51:57+00:00
Evidence text: Active Jira dependency: upstream issue ENTESB-13841 blocks downstream issue ENTESB-11455; link active since 2020-05-28T07:51:57+00:00.


In [14]:
from IPython.display import display, Markdown

# Combine original and LLM-prepared evidence
all_evidence = {}

for e in request.get("evidence", []):
    if isinstance(e, dict):
        all_evidence[str(e.get("evidence_id"))] = e

for e in prepared.get(report["trigger_id"], []):
    if isinstance(e, dict):
        all_evidence[str(e.get("evidence_id"))] = e

display(Markdown("# GraphRAG Risk Investigation Demo"))

display(Markdown("## 1. Investigation Input"))
print("Project:", report["instance_id"])
print("Checkpoint:", report["checkpoint_time"])
print("Risk types:", ", ".join(report["risk_types"]))
print("Model:", report["generation_mode"])

display(Markdown("## 2. Retrieved Historical Evidence"))

cited_ids = list(dict.fromkeys(
    eid
    for claim in report.get("claims", [])
    for eid in claim.get("evidence_ids", [])
))

for eid in cited_ids:
    e = all_evidence.get(eid)

    if e:
        print("\nEvidence ID:", eid)
        print("Source:", e.get("source_type"))
        print("Available at:", e.get("available_at"))
        print("Evidence:", e.get("text"))
    else:
        print("\nUNRESOLVED CITATION:", eid)

display(Markdown("## 3. Qwen-Generated Investigation"))

print("Summary:", report.get("summary", ""))

for i, claim in enumerate(report.get("claims", []), 1):
    print(f"\nObservation {i}: {claim.get('claim')}")
    print("Citations:", ", ".join(claim.get("evidence_ids", [])))

display(Markdown("## 4. Conclusion"))
print(report.get("conclusion", ""))

display(Markdown("## 5. Limitations"))

caveats = report.get("caveats", "")
if isinstance(caveats, str):
    print(caveats)
elif isinstance(caveats, list):
    for c in caveats:
        print("-", c)

print("\nCitation coverage:",
      sum(eid in all_evidence for eid in cited_ids),
      "/", len(cited_ids))

# GraphRAG Risk Investigation Demo

## 1. Investigation Input

Project: ENTESB-I01
Checkpoint: 2020-08-26T07:32:26+00:00
Risk types: dependency, scope
Model: LOCAL_LLM


## 2. Retrieved Historical Evidence


Evidence ID: DEP::ENTESB-I01::20::ENTESB-13841::ENTESB-11455
Source: JIRA
Available at: 2020-05-28T07:51:57+00:00
Evidence: Active Jira dependency: upstream issue ENTESB-13841 blocks downstream issue ENTESB-11455; link active since 2020-05-28T07:51:57+00:00.

Evidence ID: DEP::ENTESB-I01::10::ENTESB-13841::ENTESB-11455
Source: JIRA
Available at: 2020-05-28T07:51:57+00:00
Evidence: Active Jira dependency: upstream issue ENTESB-13841 blocks downstream issue ENTESB-11455; link active since 2020-05-28T07:51:57+00:00.

Evidence ID: DEP::ENTESB-I01::20::ENTESB-13841::ENTESB-13085
Source: JIRA
Available at: 2020-05-28T07:51:08+00:00
Evidence: Active Jira dependency: upstream issue ENTESB-13841 blocks downstream issue ENTESB-13085; link active since 2020-05-28T07:51:08+00:00.

Evidence ID: DEP::ENTESB-I01::10::ENTESB-13841::ENTESB-13085
Source: JIRA
Available at: 2020-05-28T07:51:08+00:00
Evidence: Active Jira dependency: upstream issue ENTESB-13841 blocks downstream issue ENTESB-13085; link 

...

Summary: There is limited evidence of active dependencies and status changes in JIRA issues related to project risks. Dependency chains and status transitions are observed but not conclusively linked to specific project risks.

Observation 1: Dependency chain exists between ENTESB-13841 and ENTESB-11455
Citations: DEP::ENTESB-I01::20::ENTESB-13841::ENTESB-11455, DEP::ENTESB-I01::10::ENTESB-13841::ENTESB-11455

Observation 2: Dependency chain exists between ENTESB-13841 and ENTESB-13085
Citations: DEP::ENTESB-I01::20::ENTESB-13841::ENTESB-13085, DEP::ENTESB-I01::10::ENTESB-13841::ENTESB-13085


## 4. Conclusion

LIMITED_EVIDENCE_OBSERVATIONS


## 5. Limitations

The provided evidence is derived from JIRA status transitions and dependency chains. It does not establish causality or predict future outcomes.

Citation coverage: 4 / 4


In [15]:
import pandas as pd

rows = []

for r in reports:
    risks = r.get("risk_types", [])
    if isinstance(risks, str):
        risks = [risks]

    rows.append({
        "trigger_id": r.get("trigger_id"),
        "instance_id": r.get("instance_id"),
        "risk_types": ", ".join(risks),
        "conclusion": r.get("conclusion"),
        "generation_mode": r.get("generation_mode"),
        "claim_count": len(r.get("claims") or []),
        "citation_count": sum(
            len(c.get("evidence_ids") or [])
            for c in (r.get("claims") or [])
        )
    })

demo_candidates = pd.DataFrame(rows)

print("Reports by risk type:")
display(
    demo_candidates.groupby("risk_types")
    .size()
    .reset_index(name="reports")
    .sort_values("reports", ascending=False)
)

print("\nReports by conclusion:")
display(
    demo_candidates.groupby("conclusion")
    .size()
    .reset_index(name="reports")
)

print("\nPotential demonstration cases:")
display(
    demo_candidates[
        (demo_candidates["claim_count"] > 0)
        & (demo_candidates["citation_count"] > 0)
    ].sort_values(
        ["claim_count", "citation_count"],
        ascending=False
    ).head(15)
)

Reports by risk type:


,risk_types,reports
4,schedule,41
6,scope,40
0,dependency,27
5,"schedule, scope",11
1,"dependency, schedule",8
2,"dependency, schedule, scope",7
3,"dependency, scope",5



Reports by conclusion:


,conclusion,reports
0,INCONCLUSIVE,18
1,LIMITED_EVIDENCE_OBSERVATIONS,121



Potential demonstration cases:


,trigger_id,instance_id,risk_types,conclusion,generation_mode,claim_count,citation_count
26,6c9050f7cd2661771ad1,ENTESB-I01,schedule,LIMITED_EVIDENCE_OBSERVATIONS,LOCAL_LLM,4,8
59,f2ee5eac36b36e5911d7,AMBARI-I01,"dependency, schedule",LIMITED_EVIDENCE_OBSERVATIONS,LOCAL_LLM,4,4
134,f5c692c17b464982803e,SPARK-I01,scope,LIMITED_EVIDENCE_OBSERVATIONS,LOCAL_LLM,3,3
50,d31ca73b2ee443b97a7b,AMBARI-I01,dependency,LIMITED_EVIDENCE_OBSERVATIONS,LOCAL_LLM,2,10
25,6b6459d97820e92ad3fa,WT-I01,"schedule, scope",LIMITED_EVIDENCE_OBSERVATIONS,LOCAL_LLM,2,8
54,dc142f116cb43caa6b30,ENTESB-I01,scope,LIMITED_EVIDENCE_OBSERVATIONS,LOCAL_LLM,2,8
74,244077a603ece23e5b3a,SPARK-I01,schedule,LIMITED_EVIDENCE_OBSERVATIONS,LOCAL_LLM,2,8
75,24c44eadb232cc328147,AEROGEAR-I01,"dependency, schedule, scope",LIMITED_EVIDENCE_OBSERVATIONS,LOCAL_LLM,2,8
44,c403a180d1e2a3e99c71,AMBARI-I01,"dependency, schedule",LIMITED_EVIDENCE_OBSERVATIONS,LOCAL_LLM,2,7
81,441e38fd981aa94f7208,SPARK-I01,schedule,LIMITED_EVIDENCE_OBSERVATIONS,LOCAL_LLM,2,7


In [16]:
candidate_ids = [
    "6c9050f7cd2661771ad1",  # Schedule
    "041a60f311c0263b395c",  # Dependency
    "f5c692c17b464982803e",  # Scope
]

for tid in candidate_ids:
    r = next((x for x in reports if x.get("trigger_id") == tid), None)

    if r is None:
        print("Report not found:", tid)
        continue

    print("\n" + "=" * 65)
    print("PROJECT:", r.get("instance_id"))
    print("RISKS:", r.get("risk_types"))
    print("INVESTIGATION:", tid)
    print("SUMMARY:", r.get("summary"))

    evidence = {}

    request_match = next(
        (x for x in chosen if x.get("trigger_id") == tid),
        None
    )

    if request_match:
        for e in request_match.get("evidence", []):
            if isinstance(e, dict):
                evidence[str(e.get("evidence_id"))] = e

    for e in prepared.get(tid, []):
        if isinstance(e, dict):
            evidence[str(e.get("evidence_id"))] = e

    cited = set()

    for claim in r.get("claims", []):
        ids = claim.get("evidence_ids") or []
        cited.update(ids)

        print("\nCLAIM:", claim.get("claim"))
        print("CITATIONS:", ids)

    missing = cited - set(evidence)

    print("\nCITATION COVERAGE:",
          len(cited) - len(missing), "/", len(cited))

    if missing:
        print("UNRESOLVED CITATIONS:", sorted(missing))

    print("CONCLUSION:", r.get("conclusion"))


PROJECT: ENTESB-I01
RISKS: ['schedule']
INVESTIGATION: 6c9050f7cd2661771ad1
SUMMARY: There have been several Jira status transitions that indicate issues were reopened after being resolved or closed. However, the evidence is derived from checkpoint states and does not provide direct causal links to schedule risks.

CLAIM: Jira issue ENTESB-13255 was Reopened after being Resolved.
CITATIONS: ['STATUS_CHANGE::ENTESB-I01::2::ENTESB-13255', 'JIRA_STATE::ENTESB-I01::2::ENTESB-13255']

CLAIM: Jira issue ENTESB-13381 was Reopened after being Resolved.
CITATIONS: ['STATUS_CHANGE::ENTESB-I01::2::ENTESB-13381', 'JIRA_STATE::ENTESB-I01::2::ENTESB-13381']

CLAIM: Jira issue ENTESB-13256 was Closed after being Resolved.
CITATIONS: ['STATUS_CHANGE::ENTESB-I01::2::ENTESB-13256', 'JIRA_STATE::ENTESB-I01::2::ENTESB-13256']

CLAIM: Jira issue ENTESB-13368 was Closed after being In Productization.
CITATIONS: ['STATUS_CHANGE::ENTESB-I01::2::ENTESB-13368', 'JIRA_STATE::ENTESB-I01::2::ENTESB-13368']

CITAT

In [17]:
tid = "6c9050f7cd2661771ad1"

r = next(x for x in reports if x["trigger_id"] == tid)
req = next(x for x in chosen if x["trigger_id"] == tid)

evidence = {}

for e in req.get("evidence", []):
    if isinstance(e, dict):
        evidence[str(e.get("evidence_id"))] = e

for e in prepared.get(tid, []):
    if isinstance(e, dict):
        evidence[str(e.get("evidence_id"))] = e

for i, claim in enumerate(r.get("claims", []), 1):
    print(f"\nOBSERVATION {i}")
    print(claim.get("claim"))

    for eid in claim.get("evidence_ids", []):
        e = evidence.get(eid)

        if e:
            print("\nFOUND:", eid)
            print("Evidence:", e.get("text"))
            print("Available at:", e.get("available_at"))
        else:
            print("\nNOT FOUND:", eid)


OBSERVATION 1
Jira issue ENTESB-13255 was Reopened after being Resolved.

FOUND: STATUS_CHANGE::ENTESB-I01::2::ENTESB-13255
Evidence: Observed Jira status transition for issue ENTESB-13255: Resolved -> Reopened; change occurred after 2020-04-15T07:32:26+00:00 and by 2020-04-22T07:32:26+00:00.
Available at: 2020-04-22T07:32:26+00:00

FOUND: JIRA_STATE::ENTESB-I01::2::ENTESB-13255
Evidence: Jira issue ENTESB-13255: historical status at checkpoint is Reopened.
Available at: 2020-04-22T07:32:26+00:00

OBSERVATION 2
Jira issue ENTESB-13381 was Reopened after being Resolved.

FOUND: STATUS_CHANGE::ENTESB-I01::2::ENTESB-13381
Evidence: Observed Jira status transition for issue ENTESB-13381: Resolved -> Reopened; change occurred after 2020-04-15T07:32:26+00:00 and by 2020-04-22T07:32:26+00:00.
Available at: 2020-04-22T07:32:26+00:00

FOUND: JIRA_STATE::ENTESB-I01::2::ENTESB-13381
Evidence: Jira issue ENTESB-13381: historical status at checkpoint is Reopened.
Available at: 2020-04-22T07:32:26+